In [ ]:
!pip install "figuard[openai-agents]" openai-agents -q
print("✓ FiGuard + OpenAI Agents SDK installed")

In [ ]:
from agents import Agent, Runner, function_tool
from figuard.integrations.openai_agents import guarded_function_tool
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://sandbox.figuard.io",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="travel booking session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

In [ ]:
# @guarded_function_tool wraps the raw function BEFORE @function_tool converts
# it to a tool schema — decorator order matters.

@function_tool
@guarded_function_tool(
    client=client,
    session_token=budget.session_token,
    category="flight",
    amount_key="amount",   # the kwarg that holds the spend amount
    agent_id="travel_agent",
)
def book_flight(description: str, amount: float) -> str:
    """Book a flight for the given description and amount."""
    # In production: call your booking API here
    return f"Flight booked: {description} for ${amount}"

agent = Agent(
    name="travel_agent",
    instructions="Help the user book flights within their budget.",
    tools=[book_flight],
)

# result = Runner.run_sync(agent, "Book me a flight from SFO to JFK for $270")
# Uncomment the line above to run — set OPENAI_API_KEY first.

print("✓ Agent defined with FiGuard-gated tool")
print("  If the tool call exceeds the budget, the agent receives a denial string")
print("  and FiGuard records the attempt without consuming any budget.")
print(f"\n→ See the spend tree: https://sandbox.figuard.io/ui")